# SSTW Wan2.1 Flow Adapter Preflight - Colab 冷启动入口

该 Notebook 面向从空环境启动的 Colab。它会挂载 Google Drive、拉取仓库代码、安装依赖、下载 Wan2.1 权重、运行真实 GPU preflight, 并把结果打包保存到 Google Drive。

正式 records、artifacts 和 package manifest 均由仓库模块生成, Notebook 只作为入口。


## 阶段边界

本入口只验证: Wan2.1 能否加载、callback 是否捕获 latent、time grid 是否记录、sampler signature 是否记录、velocity / latent displacement proxy 是否可获得、L4 显存是否足够。

若 `adapter_preflight_decision != PASS`, 必须先修 adapter, 不得进入 正式 generative video paper workflow。


In [ ]:
# 1. 挂载 Google Drive 并检查 GPU
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys, json
from pathlib import Path

print(subprocess.getoutput('nvidia-smi'))
DRIVE_PROJECT_ROOT = '/content/drive/MyDrive/SSTW'
Path(DRIVE_PROJECT_ROOT).mkdir(parents=True, exist_ok=True)
print('drive_project_root =', DRIVE_PROJECT_ROOT)


In [ ]:
# 2. 冷启动获取仓库代码
# 如果使用私有 fork, 请把 REPO_URL 改成你自己的可访问地址。
REPO_URL = 'https://github.com/RICHAAARC/SSTW.git'
REPO_DIR = '/content/SSTW'

if not Path(REPO_DIR).exists():
    if not REPO_URL:
        raise RuntimeError('请先设置 REPO_URL, 或将仓库上传到 /content/SSTW')
    !git clone "$REPO_URL" "$REPO_DIR"
else:
    print('仓库目录已存在, 执行 git pull 以同步远端代码。')
    %cd "$REPO_DIR"
    !git pull --ff-only
%cd "$REPO_DIR"


In [ ]:
# 3. 安装 Colab GPU 运行依赖
# 使用 diffusers GitHub 版本, 以提高 WanPipeline 可用概率。
%pip install -U git+https://github.com/huggingface/diffusers transformers accelerate safetensors imageio imageio-ffmpeg sentencepiece protobuf huggingface_hub


In [ ]:
# 4. Hugging Face 认证
# 推荐在 Colab Secrets 中配置 HF_TOKEN。此处只读取环境变量, 不打印 token 值, 不写入 records。
import os
from huggingface_hub import login

if os.environ.get('HF_TOKEN'):
    login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
    print('HF_TOKEN 状态: provided from environment.')
else:
    print('HF_TOKEN 状态: not_provided in environment, 将尝试匿名下载公开模型。')


In [ ]:
# 5. 初始化 Drive 归档布局, 并切换本地阶段 workspace
from paper_workflow.notebook_utils import flow_model_adapter_preflight_workflow as preflight_workflow
from workflows.stage_package_sync import prepare_colab_stage_layout, publish_colab_stage_package

os.environ.setdefault('SSTW_COLAB_STAGE_IO_MODE', 'local_zip')
drive_layout = preflight_workflow.ensure_drive_layout(DRIVE_PROJECT_ROOT)
layout = prepare_colab_stage_layout(
    drive_layout,
    notebook_role='wan21_flow_adapter_preflight',
)
print(json.dumps({
    'drive_layout': drive_layout,
    'active_local_layout': layout,
}, ensure_ascii=False, indent=2))


In [ ]:
# 6. 先运行默认测试和 harness 审计, 确认 Colab 中仓库状态满足治理约束。
!pytest -q
!python tools/harness/run_all_audits.py


In [ ]:
# 7. 配置真实 Wan2.1 GPU preflight
MODEL_ID = preflight_workflow.DEFAULT_WAN21_PREFLIGHT_MODEL_ID
NUM_INFERENCE_STEPS = 4
NUM_FRAMES = 33
HEIGHT = 320
WIDTH = 512

print('MODEL_ID =', MODEL_ID)
print('preflight output =', layout['drive_run_root'])


In [ ]:
# 8. 执行真实 GPU preflight
# 正式 records / artifacts 会由 experiments 模块写入 Drive run 子目录。
cmd = preflight_workflow.build_wan21_flow_adapter_preflight_command(
    layout,
    model_id=MODEL_ID,
    num_inference_steps=NUM_INFERENCE_STEPS,
    num_frames=NUM_FRAMES,
    height=HEIGHT,
    width=WIDTH,
)
print(' '.join(cmd))
result = preflight_workflow.run_command(cmd)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise SystemExit(result.returncode)


In [ ]:
# 9. 检查 preflight decision。失败时停止, 不进入正式 generative video paper workflow。
decision_path = Path(layout['drive_run_root']) / 'artifacts' / 'wan21_flow_adapter_preflight_decision.json'
decision = json.loads(decision_path.read_text(encoding='utf-8'))
print(json.dumps(decision, ensure_ascii=False, indent=2, sort_keys=True))

assert decision['adapter_preflight_decision'] == 'PASS', 'preflight 未通过, 不得进入 sampling_time_constraint_probe sampling-time constraint'
assert decision['model_load_status'] == 'loaded'
assert decision['callback_latent_capture_status'] == 'captured'
assert decision['time_grid_capture_status'] == 'captured'
assert decision['sampler_signature_status'] == 'captured'
assert decision['velocity_proxy_status'] == 'captured'


In [ ]:
# 10. 发布到 Google Drive helper
# 打包逻辑由仓库脚本执行, Notebook 不直接创建 package manifest。
cmd = preflight_workflow.build_drive_packaging_command(layout)
print(' '.join(cmd))
result = preflight_workflow.run_command(cmd)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise SystemExit(result.returncode)

stage_package = publish_colab_stage_package(
    layout,
    notebook_role='wan21_flow_adapter_preflight',
    include_videos=False,
)
print(json.dumps(stage_package, ensure_ascii=False, indent=2))


In [ ]:
# 11. 显示可下载阶段包文件。Colab 断开后, latest 阶段包仍保存在 Google Drive。
stage_package_dir = Path(layout['stage_package_dir'])
print('stage_package_dir =', stage_package_dir)
for key in ('drive_stage_package_zip', 'stage_package_manifest_path'):
    path_text = stage_package.get(key, '')
    path = Path(path_text) if path_text else None
    print(key, 'exists =', bool(path and path.exists()), 'path =', path_text)
